# Exercise 5: Launching AWS RDS PostgreSQL

In this exercise, you will learn how to:
1. Use the AWS CLI to launch and connect to an AWS RDS instance running PostgreSQL
2. Store and retrieve database credentials securely using AWS Secrets Manager
3. Connect to and query your RDS PostgreSQL database
4. Clean up resources to avoid unnecessary charges

## Prerequisites
- AWS Account with appropriate permissions
- AWS CLI installed and configured
- Basic understanding of PostgreSQL and SQL

## Step 1: Install and Configure AWS CLI

First, let's verify that the AWS CLI is installed and configured properly. If you haven't installed it yet, you can download it from [AWS CLI Installation Guide](https://aws.amazon.com/cli/).

In [ ]:
# Verify AWS CLI installation
!aws --version

In [ ]:
# Check AWS CLI configuration
!aws sts get-caller-identity

## Step 2: Create AWS Secrets for Database Credentials

AWS Secrets Manager helps you protect access to your applications, services, and IT resources. Instead of hardcoding credentials in your code, you can store them securely in Secrets Manager.

In [ ]:
import json
import random
import string

# Generate a random password for the database
def generate_password(length=16):
    characters = string.ascii_letters + string.digits + "!@#$%^&*"
    return ''.join(random.choice(characters) for i in range(length))

# Database credentials
db_username = "dbadmin"
db_password = generate_password()

# Create a secret structure
secret_dict = {
    "username": db_username,
    "password": db_password
}

print("Generated credentials (save these temporarily):")
print(f"Username: {db_username}")
print(f"Password: {db_password}")

In [ ]:
# Store the secret in AWS Secrets Manager
secret_name = "rds-postgres-credentials"

# Create the secret using AWS CLI
import subprocess

create_secret_cmd = f'''aws secretsmanager create-secret \
    --name {secret_name} \
    --description "RDS PostgreSQL credentials for exercise" \
    --secret-string '{json.dumps(secret_dict)}' '''

result = subprocess.run(create_secret_cmd, shell=True, capture_output=True, text=True)
print("Secret creation output:")
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

## Step 3: Launch RDS PostgreSQL Instance

Now we'll use the AWS CLI to create an RDS instance running PostgreSQL. This process may take 5-10 minutes to complete.

In [ ]:
# Define RDS instance parameters
db_instance_identifier = "postgres-exercise-db"
db_instance_class = "db.t3.micro"  # Free tier eligible
allocated_storage = 20  # GB
engine = "postgres"
engine_version = "14.7"
db_name = "exercisedb"

# Create RDS instance
create_rds_cmd = f'''aws rds create-db-instance \
    --db-instance-identifier {db_instance_identifier} \
    --db-instance-class {db_instance_class} \
    --engine {engine} \
    --engine-version {engine_version} \
    --master-username {db_username} \
    --master-user-password {db_password} \
    --allocated-storage {allocated_storage} \
    --db-name {db_name} \
    --publicly-accessible \
    --backup-retention-period 0 \
    --no-multi-az \
    --storage-type gp2'''

result = subprocess.run(create_rds_cmd, shell=True, capture_output=True, text=True)
print("RDS Instance creation initiated:")
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

In [ ]:
# Check the status of the RDS instance
import time

def check_rds_status(db_identifier):
    """Check the status of the RDS instance"""
    cmd = f"aws rds describe-db-instances --db-instance-identifier {db_identifier}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        data = json.loads(result.stdout)
        status = data['DBInstances'][0]['DBInstanceStatus']
        return status
    return None

print("Checking RDS instance status...")
status = check_rds_status(db_instance_identifier)
print(f"Current status: {status}")
print("\nNote: The instance typically takes 5-10 minutes to become 'available'.")

## Step 4: Retrieve Database Endpoint

Once the RDS instance is available, we need to retrieve the endpoint URL that we'll use to connect to the database.

In [ ]:
# Wait for the instance to be available (optional: use this to wait automatically)
def wait_for_rds_available(db_identifier, max_wait_time=600, check_interval=30):
    """Wait for the RDS instance to become available"""
    elapsed_time = 0
    print("Waiting for RDS instance to become available...")
    
    while elapsed_time < max_wait_time:
        status = check_rds_status(db_identifier)
        print(f"Status after {elapsed_time}s: {status}")
        
        if status == "available":
            print("RDS instance is now available!")
            return True
        
        time.sleep(check_interval)
        elapsed_time += check_interval
    
    print("Timeout waiting for RDS instance.")
    return False

# Uncomment the line below to wait for the instance to be ready
# wait_for_rds_available(db_instance_identifier)

In [ ]:
# Retrieve the endpoint once the instance is available
def get_rds_endpoint(db_identifier):
    """Get the endpoint address of the RDS instance"""
    cmd = f"aws rds describe-db-instances --db-instance-identifier {db_identifier}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        data = json.loads(result.stdout)
        endpoint = data['DBInstances'][0]['Endpoint']['Address']
        port = data['DBInstances'][0]['Endpoint']['Port']
        return endpoint, port
    return None, None

# Get the endpoint
db_endpoint, db_port = get_rds_endpoint(db_instance_identifier)
if db_endpoint:
    print(f"Database Endpoint: {db_endpoint}")
    print(f"Port: {db_port}")
else:
    print("Unable to retrieve endpoint. Instance may not be available yet.")

## Step 5: Retrieve Credentials from AWS Secrets Manager

Now let's retrieve the credentials we stored in AWS Secrets Manager.

In [ ]:
# Retrieve the secret from AWS Secrets Manager
def get_secret(secret_name):
    """Retrieve secret from AWS Secrets Manager"""
    cmd = f"aws secretsmanager get-secret-value --secret-id {secret_name}"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    if result.returncode == 0:
        secret_data = json.loads(result.stdout)
        secret_string = json.loads(secret_data['SecretString'])
        return secret_string
    return None

# Get the credentials
credentials = get_secret(secret_name)
if credentials:
    retrieved_username = credentials['username']
    retrieved_password = credentials['password']
    print("Successfully retrieved credentials from AWS Secrets Manager!")
    print(f"Username: {retrieved_username}")
else:
    print("Failed to retrieve credentials.")

## Step 6: Connect to RDS PostgreSQL Instance

Now we'll connect to the RDS PostgreSQL database using the `psycopg2` library.

In [ ]:
# Install psycopg2 if not already installed
!pip install psycopg2-binary

In [ ]:
import psycopg2
from psycopg2 import Error

# Connect to the PostgreSQL database
try:
    connection = psycopg2.connect(
        host=db_endpoint,
        port=db_port,
        database=db_name,
        user=retrieved_username,
        password=retrieved_password
    )
    
    print("Successfully connected to PostgreSQL RDS instance!")
    print(f"PostgreSQL server information:")
    print(connection.get_dsn_parameters())
    
except (Exception, Error) as error:
    print("Error while connecting to PostgreSQL:", error)

## Step 7: Verify Database Connection

Let's verify the connection by checking the PostgreSQL version and running a simple query.

In [ ]:
# Create a cursor and execute a query
cursor = connection.cursor()

# Check PostgreSQL version
cursor.execute("SELECT version();")
db_version = cursor.fetchone()
print("PostgreSQL database version:")
print(db_version[0])

## Step 8: Query the Database

Let's create a simple table and insert some data to test our database connection.

In [ ]:
# Create a sample table
create_table_query = '''
CREATE TABLE IF NOT EXISTS students (
    student_id SERIAL PRIMARY KEY,
    name VARCHAR(100) NOT NULL,
    email VARCHAR(100) UNIQUE NOT NULL,
    enrollment_date DATE DEFAULT CURRENT_DATE
);
'''

cursor.execute(create_table_query)
connection.commit()
print("Table 'students' created successfully!")

In [ ]:
# Insert sample data
insert_query = '''
INSERT INTO students (name, email) VALUES
    ('Alice Johnson', 'alice.johnson@example.com'),
    ('Bob Smith', 'bob.smith@example.com'),
    ('Carol Williams', 'carol.williams@example.com')
ON CONFLICT (email) DO NOTHING;
'''

cursor.execute(insert_query)
connection.commit()
print(f"{cursor.rowcount} records inserted successfully!")

In [ ]:
# Query the data
select_query = "SELECT * FROM students ORDER BY student_id;"
cursor.execute(select_query)
records = cursor.fetchall()

print("Students in the database:")
print(f"{'ID':<5} {'Name':<20} {'Email':<30} {'Enrollment Date'}")
print("-" * 75)
for record in records:
    print(f"{record[0]:<5} {record[1]:<20} {record[2]:<30} {record[3]}")

## Step 9: Clean Up Resources

**IMPORTANT:** To avoid unnecessary AWS charges, make sure to clean up the resources you created in this exercise.

In [ ]:
# Close the database connection
if connection:
    cursor.close()
    connection.close()
    print("PostgreSQL connection closed.")

In [ ]:
# Delete the RDS instance
delete_rds_cmd = f'''aws rds delete-db-instance \
    --db-instance-identifier {db_instance_identifier} \
    --skip-final-snapshot'''

result = subprocess.run(delete_rds_cmd, shell=True, capture_output=True, text=True)
print("RDS instance deletion initiated:")
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)

In [ ]:
# Delete the secret from AWS Secrets Manager
delete_secret_cmd = f"aws secretsmanager delete-secret --secret-id {secret_name} --force-delete-without-recovery"

result = subprocess.run(delete_secret_cmd, shell=True, capture_output=True, text=True)
print("Secret deletion initiated:")
print(result.stdout)
if result.stderr:
    print("Errors:", result.stderr)